In [ ]:
import pandas as pd
import numpy as np
import random as rd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('/Users/subasrees/Desktop/Machine_learning_practice/Datasets/Debernardi_et_al_2020_data.csv')
df

In [ ]:
df=df.drop(['sample_id'],axis=1)
df.isnull().sum()

In [ ]:
threshold = 0.3
df = df[df.columns[df.isna().mean() < threshold]]
df.isnull().sum()

In [ ]:
y=df['diagnosis']
print(y.value_counts())
X=df.drop(['diagnosis'],axis=1)

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    f1_score, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc
)
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
def plot_confmat(
    y_true,
    y_pred,
    title="Confusion matrix",
    savepath=None,
    show=True
):
    labels = np.unique(y_true)

    fig, ax = plt.subplots(figsize=(4.8, 3.8))
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=labels
    )
    disp.plot(values_format="d", ax=ax)
    ax.set_xlabel("Predicted label", fontsize=11)
    ax.set_ylabel("True label", fontsize=11)
    ax.set_title(title)
    plt.xticks(rotation=30,ha='right',fontsize=10)
    plt.yticks(rotation=0, fontsize=10)
    plt.tight_layout()

    # SAVE
    if savepath is not None:
        os.makedirs(savepath, exist_ok=True)
        fname = title.replace(" ", "_").lower()
        plt.savefig(
        os.path.join(savepath, f"{fname}.pdf"),
        dpi=300,
        bbox_inches="tight")
        os.makedirs(savepath, exist_ok=True)
        fname = title.replace(" ", "_").lower()
        plt.savefig(
        os.path.join(savepath, f"{fname}.jpg"),
        dpi=300,
        bbox_inches="tight")

    # SHOW OR CLOSE
    if show:
        plt.show()
    else:
        plt.close(fig)


In [ ]:
def plot_roc(model, X_test, y_test, title="ROC",savepath=None,show=True):
    classes = np.unique(y_test)
    n_classes = len(classes)

    # Must have predict_proba for ROC
    if not hasattr(model, "predict_proba"):
        print(f"[ROC skipped] {title}: model has no predict_proba")
        return

    y_prob = model.predict_proba(X_test)

    plt.figure(figsize=(7,5))

    if n_classes == 2:
        # positive class = classes[1]
        y_true_bin = (y_test == classes[1]).astype(int)
        fpr, tpr, _ = roc_curve(y_true_bin, y_prob[:, 1])
        roc_auc = auc(fpr, tpr)

        plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
        plt.plot([0,1],[0,1], linestyle="--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(title)
        plt.legend()
        plt.tight_layout()
        plt.show()
        return

    # Multiclass: one-vs-rest
    y_bin = label_binarize(y_test, classes=classes)   # shape (n_samples, n_classes)

    for i, cls in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{cls} (AUC={roc_auc:.3f})")

    plt.plot([0,1],[0,1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    #plt.show()
    # SAVE
    if savepath is not None:
        os.makedirs(savepath, exist_ok=True)
        fname = title.replace(" ", "_").lower()
        plt.savefig(
            os.path.join(savepath, f"{fname}.pdf"),
            dpi=300,
            bbox_inches="tight")
        plt.savefig(
            os.path.join(savepath, f"{fname}.jpg"),
            dpi=300,
            bbox_inches="tight")

    # SHOW OR CLOSE
    if show:
        plt.show()
    else:
        plt.close(fig)



In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
categorical_columns=['patient_cohort','sample_origin','sex']
numerical_columns=list(set(X.columns)-set(categorical_columns))
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('cat', categorical_transformer, categorical_columns),
     ('num', StandardScaler(), numerical_columns)
])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [ ]:
from sklearn.preprocessing import LabelEncoder

# 1) Fit encoder on y_train
le = LabelEncoder()
y_train_full_enc = le.fit_transform(y_train)
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)  # must transform test too

# Check
print(le.classes_)
print(np.unique(y_train_enc))

In [ ]:
models = {
    'RF': RandomForestClassifier(),
    'GB': GradientBoostingClassifier(),
    'HGB': HistGradientBoostingClassifier(),
    'LGBM': LGBMClassifier(),
    'XGB':XGBClassifier()
}

In [ ]:
param_grids = {
    "RF": {
        "model__n_estimators": [300, 500],
        "model__max_depth": [None, 10, 20]
    },
    "GB": {
        "model__learning_rate": [0.05, 0.1],
        "model__n_estimators": [200, 400],
        "model__max_depth": [2, 3]
    },
    "HGB": {
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_depth": [4, 6, None],
        "model__max_iter": [200, 400]
    },
    "LGBM": {
        "model__n_estimators": [200, 400],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [-1, 4, 6],     # -1 = no limit
        "model__subsample": [0.8],
        "model__colsample_bytree": [0.8],
        "model__num_leaves": [31, 63],
    },
    "XGB" : {
        "model__n_estimators": [200, 400],
        "model__max_depth": [4, 6],
        "model__learning_rate": [0.05, 0.1],
        "model__subsample": [0.8],
        "model__colsample_bytree": [0.8]
}
}


In [ ]:
model_names=['RF','GB','HGB','LGBM','XGB']

In [ ]:
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
best_models = {}
rank_pipe={}
for name, base_model in models.items():
    base_pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', base_model)
    ])
    grid = param_grids.get(name, None)

    if grid is None:
        print(f"\n{name}: no param grid provided, skipping tuning.")
        best_models[name] = base_pipe
        continue

    print("\n" + "="*70)
    print("Tuning:", name)

    gs = GridSearchCV(
        base_pipe,
        param_grid=grid,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1
    )
    if name=='XGB':
        gs.fit(X_train, y_train_full_enc)
    else:
        gs.fit(X_train, y_train)

    print("Best CV macro-F1:", round(gs.best_score_, 4))
    print("Best params:", gs.best_params_)

    best_models[name] = gs.best_estimator_
    rank_pipe[name] = {"pipe": base_pipe, "f1_macro": round(gs.best_score_, 4)}


In [ ]:
# Extract F1 scores
f1_values = {name: info['f1_macro'] for name, info in rank_pipe.items()}

# Sort by F1 descending and take top 3
top_3 = dict(sorted(f1_values.items(), key=lambda x: x[1], reverse=True)[:3])

print(top_3)

In [ ]:
top3_models={}
for i,j in top_3.items():
    top3_models[i]=rank_pipe[i]

In [ ]:
from sklearn.metrics import roc_auc_score
#savepath='/Users/subasrees/Pediatric-cancer-analysis/Figs/'
for name, model_dict in top3_models.items():
    model_pipe = model_dict['pipe']
    if name=='XGB':
        model_pipe.fit(X_train, y_train_enc)
        y_pred_enc = model_pipe.predict(X_test)
        y_prob_enc = model_pipe.predict_proba(X_test)
        mf1 = f1_score(y_test_enc, y_pred_enc, average="macro")
        print("\n" + "="*70)
        print(name, "TEST macro-F1 =", round(mf1, 4))
        macro_auc = roc_auc_score(
        y_test_enc,
        y_prob_enc,
        multi_class="ovr",
        average="macro")
        print(name,"Macro AUC:", round(macro_auc,4))
        plot_confmat(y_test_enc, y_pred_enc, title=f"{name} | Confusion matrix (test)", savepath=None,show=True)
        plot_roc(model_pipe, X_test, y_test_enc, title=f"{name} ROC (test)",savepath=None,show=True)
    else:
        model_pipe.fit(X_train, y_train)
        y_pred = model_pipe.predict(X_test)
        y_prob = model_pipe.predict_proba(X_test)
        mf1 = f1_score(y_test, y_pred, average="macro")
        print("\n" + "="*70)
        print(name, "TEST macro-F1 =", round(mf1, 4))
        macro_auc = roc_auc_score(
        y_test,
        y_prob,
        multi_class="ovr",
        average="macro")
        print(name,"Macro AUC:", round(macro_auc,4))
        plot_confmat(y_test, y_pred, title=f"{name} | Confusion matrix (test)", savepath=None,show=True)
        plot_roc(model_pipe, X_test, y_test, title=f"{name} ROC (test)",savepath=None,show=True)



In [ ]:
def shap_any_model_multiclass_with_importance(
    model,                    # ← model ONLY (not pipeline)
    X_train_df: pd.DataFrame, # ← transformed numeric
    X_test_df: pd.DataFrame,  # ← transformed numeric
    class_names,
    model_name: str,
    background_size: int = 100,
    random_state: int = 42,
    make_plots: bool = False,
    savepath: str | None = None,
    max_evals: int = 2000,
):
    import numpy as np
    import pandas as pd
    import shap
    import matplotlib.pyplot as plt
    import os

    class_names = list(class_names)
    feat_names = list(X_test_df.columns)

    rng = np.random.default_rng(random_state)

    # Background subset
    bg_n = min(background_size, len(X_train_df))
    bg_idx = rng.choice(len(X_train_df), size=bg_n, replace=False)
    X_bg = X_train_df.iloc[bg_idx]

    # ✅ Use model ONLY
    explainer = shap.Explainer(model.predict_proba, X_bg)
    shap_exp = explainer(X_test_df, max_evals=max_evals)

    if shap_exp.values.ndim != 3:
        raise ValueError(f"Expected (n,f,c), got {shap_exp.values.shape}")

    n, f, c = shap_exp.values.shape

    if c != len(class_names):
        raise ValueError(f"Class mismatch: SHAP has {c}, class_names has {len(class_names)}")

    # ---------- Global importance ----------
    global_vals = np.mean(np.abs(shap_exp.values), axis=(0, 2))
    imp_global = pd.Series(global_vals, index=feat_names).sort_values(ascending=False)

    # ---------- Class-specific importance ----------
    imp_per_class = {}

    for k, cls in enumerate(class_names):
        class_vals = np.mean(np.abs(shap_exp.values[:, :, k]), axis=0)
        imp_per_class[cls] = pd.Series(class_vals, index=feat_names).sort_values(ascending=False)

        if make_plots:
            shap.summary_plot(
                shap_exp.values[:, :, k],
                features=X_test_df,
                feature_names=feat_names,
                show=False
            )

            plt.title(f"{model_name} | SHAP summary | {cls}")
            plt.tight_layout()
            plt.show()

    return imp_global, imp_per_class, shap_exp

In [ ]:
best_models=top3_models

In [ ]:
for name, model_info in best_models.items():

    pipe = model_info["pipe"]
    preprocessor = pipe.named_steps["preprocessor"]
    model = pipe.named_steps["model"]

    # Transform ONCE
    X_train_trans = preprocessor.transform(X_train)
    X_test_trans  = preprocessor.transform(X_test)

    feature_names = preprocessor.get_feature_names_out()

    # Convert to DataFrame
    X_train_trans = pd.DataFrame(X_train_trans, columns=feature_names)
    X_test_trans  = pd.DataFrame(X_test_trans,  columns=feature_names)

    model_imp_global, model_imp_class, model_shap = \
        shap_any_model_multiclass_with_importance(
            model,
            X_train_trans,
            X_test_trans,
            ["1","2","3"],
            model_name=name,
            make_plots=True
        )